# 01 - Ingestão, Reconciliação e Gold

Fluxo correto: Bronze (base antiga + base nova) -> Silver unificado -> Gold de modelagem.


In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
BASE_DIR = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.pipeline.schema_map import (
    load_and_normalize_workbook,
    load_and_normalize_old_base,
    reconcile_bases,
    validate_normalized_data,
)
from src.pipeline.relational_features import build_relational_features, enrich_with_relational_features
from src.features.risk_target import build_risk_dataset


In [3]:
# Caminhos bronze
new_workbook = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
old_csv = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'Bases antigas' / 'PEDE_PASSOS_DATASET_FIAP.csv'

print('new exists?', new_workbook.exists(), new_workbook)
print('old exists?', old_csv.exists(), old_csv)

new exists? True c:\Users\00157NLUC-BrenoR\Datathon\DATATHON-20260502T202225Z-3-001\DATATHON\BASE DE DADOS PEDE 2024 - DATATHON.xlsx
old exists? True c:\Users\00157NLUC-BrenoR\Datathon\DATATHON-20260502T202225Z-3-001\DATATHON\Bases antigas\PEDE_PASSOS_DATASET_FIAP.csv


In [4]:
# Bronze normalizado
df_new = load_and_normalize_workbook(str(new_workbook))
df_old = load_and_normalize_old_base(str(old_csv))

print('new rows:', len(df_new), '| anos:', sorted(df_new['ano_referencia'].unique()))
print('old rows:', len(df_old), '| anos:', sorted(df_old['ano_referencia'].unique()))

new rows: 3030 | anos: [np.int64(2022), np.int64(2023), np.int64(2024)]
old rows: 4047 | anos: [np.int64(2020), np.int64(2021), np.int64(2022)]


In [5]:
# Silver reconciliado (2022 prioriza base nova)
df_silver = reconcile_bases(df_new, df_old)
df_valid, df_invalid = validate_normalized_data(df_silver)

print('silver rows:', len(df_silver))
print('valid rows:', len(df_valid), '| invalid rows:', len(df_invalid))
display(df_silver.groupby(['ano_referencia','source_system']).size().to_frame('linhas'))

silver rows: 6217
valid rows: 5490 | invalid rows: 727


linhas
ano_referencia source_system           
2020           antiga_2020_2022    1349
2021           antiga_2020_2022    1349
2022           antiga_2020_2022     489
               nova_2022_2024       860
2023           nova_2022_2024      1014
2024           nova_2022_2024      1156

In [6]:
# Features relacionais (TbDiario, TbFaseNotaAluno, TbHistoricoNotas)
bases_antigas_dir = BASE_DIR / 'DATATHON-20260502T202225Z-3-001' / 'DATATHON' / 'Bases antigas'
rel_base = next(p for p in bases_antigas_dir.iterdir() if p.is_dir() and p.name.lower().startswith('base de dados - passos'))
df_rel_feat = build_relational_features(rel_base)
print('rel_features rows:', len(df_rel_feat))
display(df_rel_feat.head())


rel_features rows: 4792


,ra,ano_referencia,freq_registros,freq_tx_presenca,freq_tx_falta,fase_nota_media,fase_faltas_total,fase_aulas_total,hist_nota_final_media,hist_faltas_anual_media,hist_disciplinas
0,RA-1,2021,60.0,0.833333,0.166667,5.583333,24.0,202.0,NaN,0.0,1.0
1,RA-1,2022,115.0,0.921739,0.078261,5.641667,32.0,320.0,NaN,NaN,NaN
2,RA-1,2023,7.0,0.285714,0.714286,5.288462,8.0,118.0,NaN,NaN,NaN
3,RA-10,2021,60.0,0.816667,0.166667,7.416667,20.0,202.0,NaN,0.0,1.0
4,RA-10,2022,16.0,0.000000,1.000000,7.416667,20.0,202.0,NaN,NaN,NaN


In [7]:
# Enriquecimento do silver com features relacionais
df_silver_enriched = enrich_with_relational_features(df_silver, df_rel_feat)
df_valid, df_invalid = validate_normalized_data(df_silver_enriched)
print('silver enriched rows:', len(df_silver_enriched))
print('valid rows:', len(df_valid), '| invalid rows:', len(df_invalid))
extra_cols = [c for c in df_silver_enriched.columns if c.startswith(('freq_','fase_','hist_'))]
print('cols relacionais:', extra_cols)


silver enriched rows: 6217
valid rows: 5490 | invalid rows: 727
cols relacionais: ['fase_ideal', 'freq_registros', 'freq_tx_presenca', 'freq_tx_falta', 'fase_nota_media', 'fase_faltas_total', 'fase_aulas_total', 'hist_nota_final_media', 'hist_faltas_anual_media', 'hist_disciplinas']


In [8]:
# Diagnóstico rápido de nulos (silver validado)
display((df_valid.isna().mean() * 100).sort_values(ascending=False).to_frame('%_nulos').head(25))


,%_nulos
fase_ideal,100.000000
escola,78.961749
status_ativo,78.943534
ingles,78.287796
atingiu_pv,66.083789
idade,52.076503
ipp,45.482696
genero,44.808743
portugues,42.459016
matematica,42.440801


In [9]:
# Gold inicial de risco
risk_df = build_risk_dataset(df_valid).copy()
print('risk_df shape:', risk_df.shape)
print('taxa_risco:', round(risk_df['target_risco'].mean() * 100, 2), '%')

risk_df shape: (3262, 33)
taxa_risco: 36.33 %


In [10]:
# Features baseline
features_base = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'inde', 'defasagem', 'ano_referencia']
target_col = 'target_risco'

drop_cols = ['fase_ideal', 'atingiu_pv', 'escola', 'status_ativo', 'ingles']
for c in drop_cols:
    if c in risk_df.columns:
        risk_df.drop(columns=c, inplace=True)

for c in features_base:
    if c in risk_df.columns:
        risk_df[c] = risk_df[c].fillna(risk_df[c].median())

display((risk_df[features_base + [target_col]].isna().mean() * 100).to_frame('%_nulos'))
display(risk_df[features_base + [target_col]].head())

,%_nulos
ian,0.0
ida,0.0
ieg,0.0
iaa,0.0
ips,0.0
inde,0.0
defasagem,0.0
ano_referencia,0.0
target_risco,0.0


,ian,ida,ieg,iaa,ips,inde,defasagem,ano_referencia,target_risco
0,5.0,8.8,6.3,7.5,6.9,7.319,-1.0,2021,1
1,5.0,4.0,4.1,8.3,5.6,5.783,-1.0,2022,1
2,10.0,6.5,8.6,8.8,7.5,7.360,0.0,2023,0
4,5.0,6.5,8.6,8.8,7.5,7.360,-1.0,2021,0
6,5.0,0.0,0.0,8.5,3.8,3.117,-2.0,2021,1


In [11]:
# Persistência silver e gold
silver_dir = PROJECT_ROOT / 'data' / 'silver'
gold_dir = PROJECT_ROOT / 'data' / 'gold'
silver_dir.mkdir(parents=True, exist_ok=True)
gold_dir.mkdir(parents=True, exist_ok=True)

silver_file = silver_dir / 'base_unificada_validada_enriquecida.csv'
invalid_file = silver_dir / 'base_unificada_invalidos.csv'
gold_file = gold_dir / 'base_modelagem_risco.csv'

df_valid.to_csv(silver_file, index=False)
df_invalid.to_csv(invalid_file, index=False)
risk_df.to_csv(gold_file, index=False)

print('silver:', silver_file)
print('invalid:', invalid_file)
print('gold:', gold_file)



silver: c:\Users\00157NLUC-BrenoR\Datathon\datathon_passos\data\silver\base_unificada_validada_enriquecida.csv
invalid: c:\Users\00157NLUC-BrenoR\Datathon\datathon_passos\data\silver\base_unificada_invalidos.csv
gold: c:\Users\00157NLUC-BrenoR\Datathon\datathon_passos\data\gold\base_modelagem_risco.csv
